# Malware Classification

## What This Is
Static malware analysis classifies executables without running them by examining:
- **PE headers**: Machine type, linker version, compilation timestamp, subsystem
- **Import table (IAT)**: DLLs and functions imported — CreateRemoteThread, VirtualAllocEx, WriteProcessMemory signal injection
- **Section characteristics**: Executable sections, entropy per section
- **Strings**: Registry keys, IP addresses, URLs, suspicious constants
- **Entropy**: High entropy sections indicate packing/encryption

## Why GBMs Dominate Static Analysis
Gradient Boosted Machines handle the mixed feature types (categorical DLL names, numerical entropy, boolean section flags) without feature normalization. They're interpretable via feature importance and fast at inference. The Microsoft Malware Classification Challenge (2015) was won by GBM approaches.

In [ ]:
import numpy as np
import hashlib
from typing import Dict, List, Tuple

np.random.seed(42)

# Simulate PE header features for malware classification
# Real tool: pefile library for Windows PE parsing

MALICIOUS_IMPORTS = [
    'CreateRemoteThread', 'VirtualAllocEx', 'WriteProcessMemory',
    'SetWindowsHookEx', 'RegSetValueEx', 'InternetConnect',
    'URLDownloadToFile', 'ShellExecute', 'CreateService'
]

BENIGN_IMPORTS = [
    'CreateFile', 'ReadFile', 'WriteFile', 'CloseHandle',
    'GetSystemInfo', 'HeapAlloc', 'MessageBox', 'RegOpenKey',
    'LoadLibrary', 'GetProcAddress'
]

def generate_pe_features(is_malware: bool, n: int) -> np.ndarray:
    """Simulate PE header + import features."""
    samples = []
    for _ in range(n):
        if is_malware:
            # Malware tends to: high entropy, few imports, suspicious functions
            file_size = np.random.randint(20000, 500000)
            n_sections = np.random.randint(3, 8)
            entropy_code = np.random.uniform(6.5, 7.9)  # packed/encrypted
            entropy_data = np.random.uniform(5.0, 7.5)
            n_imports = np.random.randint(5, 30)
            n_mal_imports = np.random.randint(2, 8)
            has_packer_sig = np.random.binomial(1, 0.7)  # 70% are packed
            timestamp_valid = np.random.binomial(1, 0.3)  # often tampered
            n_strings = np.random.randint(50, 500)
            n_ip_strings = np.random.randint(1, 10)
            n_url_strings = np.random.randint(0, 5)
            export_count = np.random.randint(0, 3)
            tls_callback = np.random.binomial(1, 0.4)
        else:
            file_size = np.random.randint(50000, 5000000)
            n_sections = np.random.randint(4, 12)
            entropy_code = np.random.uniform(5.0, 6.5)  # normal
            entropy_data = np.random.uniform(2.0, 5.0)
            n_imports = np.random.randint(30, 200)
            n_mal_imports = np.random.randint(0, 2)
            has_packer_sig = np.random.binomial(1, 0.05)
            timestamp_valid = np.random.binomial(1, 0.9)
            n_strings = np.random.randint(200, 5000)
            n_ip_strings = np.random.randint(0, 2)
            n_url_strings = np.random.randint(0, 3)
            export_count = np.random.randint(0, 50)
            tls_callback = np.random.binomial(1, 0.02)
        
        samples.append([
            file_size, n_sections, entropy_code, entropy_data,
            n_imports, n_mal_imports, has_packer_sig, timestamp_valid,
            n_strings, n_ip_strings, n_url_strings, export_count, tls_callback
        ])
    return np.array(samples, dtype=float)

FEATURE_NAMES = [
    'file_size', 'n_sections', 'entropy_code', 'entropy_data',
    'n_imports', 'n_mal_imports', 'has_packer_sig', 'timestamp_valid',
    'n_strings', 'n_ip_strings', 'n_url_strings', 'export_count', 'tls_callback'
]

X_benign = generate_pe_features(False, 600)
X_malware = generate_pe_features(True, 400)
X = np.vstack([X_benign, X_malware])
y = np.array([0]*600 + [1]*400)

print(f'PE feature dataset: {len(X)} samples, {X.shape[1]} features')
print(f'Benign: {(y==0).sum()}, Malware: {(y==1).sum()}')


In [ ]:
# Train GBM malware classifier
# Using logistic regression as proxy (same concept, simpler implementation)

def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

# Shuffle and split
perm = np.random.permutation(len(X))
X, y = X[perm], y[perm]
split = 800
X_tr, y_tr, X_te, y_te = X[:split], y[:split], X[split:], y[split:]

mu, sig = X_tr.mean(0), X_tr.std(0) + 1
X_tr_n = (X_tr - mu) / sig
X_te_n = (X_te - mu) / sig

# Logistic regression with L2 regularization
w = np.zeros(X_tr_n.shape[1])
b = 0.0
lr, lam, epochs = 0.05, 0.01, 300

for _ in range(epochs):
    e = sigmoid(X_tr_n @ w + b) - y_tr
    w -= lr * ((X_tr_n.T @ e) / len(y_tr) + lam * w)
    b -= lr * e.mean()

proba_te = sigmoid(X_te_n @ w + b)
pred_te = (proba_te > 0.5).astype(int)

tp = ((pred_te == 1) & (y_te == 1)).sum()
fp = ((pred_te == 1) & (y_te == 0)).sum()
fn = ((pred_te == 0) & (y_te == 1)).sum()
tn = ((pred_te == 0) & (y_te == 0)).sum()

precision = tp / (tp + fp + 1e-8)
recall = tp / (tp + fn + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

print('Malware Classifier Results:')
print(f'  Accuracy:  {(pred_te == y_te).mean():.3f}')
print(f'  Precision: {precision:.3f}')
print(f'  Recall:    {recall:.3f}')
print(f'  F1:        {f1:.3f}')
print(f'  FPR:       {fp/(fp+tn+1e-8):.4f}')

# Feature importance via weight magnitude
importance = np.abs(w)
top_idx = np.argsort(importance)[::-1][:5]
print('\nTop 5 most important features:')
for i, idx in enumerate(top_idx):
    print(f'  {i+1}. {FEATURE_NAMES[idx]}: {importance[idx]:.4f}')
